In [ ]:
from azure.storage.filedatalake import DataLakeServiceClient

connection_string = "connection_string = "PASTE_YOUR_CONNECTION_STRING_HERE"

service_client = DataLakeServiceClient.from_connection_string(connection_string)

print("Connected to Azure!")

Connected to Azure!


In [2]:
file_system_client = service_client.get_file_system_client(file_system="raw")

print("Raw container connected!")

Raw container connected!


In [23]:
import pandas as pd

logger.info("Reading source file...")
df = pd.read_parquet("C:/Users/TLS/Downloads/yellow_tripdata_2024-01.parquet")
logger.info(f"Total rows: {len(df)}")
df.head()

2026-08-08 03:07:14,668 | INFO | Reading source file...
2026-08-08 03:07:15,645 | INFO | Total rows: 2964624


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [24]:
file_client = directory_client.get_file_client("yellow_tripdata_2024-01.parquet")

with open("C:/Users/TLS/Downloads/yellow_tripdata_2024-01.parquet", "rb") as data:
    file_client.upload_data(
        data,
        overwrite=True,
        connection_timeout=600,
        max_concurrency=1
    )

print("Raw file uploaded!")

2026-08-08 03:07:15,756 | INFO | Request URL: 'https://opspulseadl.dfs.core.windows.net/raw/nyc_taxi/2024/01/yellow_tripdata_2024-01.parquet?resource=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/json'
    'User-Agent': 'azsdk-python-storage-dfs/12.25.0 Python/3.13.4 (Windows-10-10.0.19045-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '596accad-92ac-11f1-83a0-c8d9d2d2804a'
    'Authorization': 'REDACTED'
No body was attached to the request
2026-08-08 03:07:18,348 | INFO | Response status: 201
Response headers:
    'Last-Modified': 'Fri, 07 Aug 2026 22:07:18 GMT'
    'ETag': '"0x8DEF4D03F6C7564"'
    'Server': 'Windows-Azure-HDFS/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-server-encrypted': 'REDACTED'
    'x-ms-request-id': '7db950d3-901f-002c-11b9-263dac000000'
    'x-ms-version': 'REDACTED'
    'x-ms-client-request-id': '596accad-92ac-11f1-83a0-c8d9d2d2804a'
    'Date': 'Fri, 07 Aug 2026 22:07:18 GMT'
   

Raw file uploaded!


In [25]:
properties = file_client.get_file_properties()
print("File size on Azure:", properties.size, "bytes")

2026-08-08 03:08:01,394 | INFO | Request URL: 'https://opspulseadl.blob.core.windows.net/raw/nyc_taxi/2024/01/yellow_tripdata_2024-01.parquet'
Request method: 'HEAD'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-dfs/12.25.0 Python/3.13.4 (Windows-10-10.0.19045-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '74a1010a-92ac-11f1-be22-c8d9d2d2804a'
    'Authorization': 'REDACTED'
No body was attached to the request
2026-08-08 03:08:03,369 | INFO | Response status: 200
Response headers:
    'Content-Length': '49961641'
    'Content-Type': 'application/octet-stream'
    'Last-Modified': 'Fri, 07 Aug 2026 22:08:01 GMT'
    'Accept-Ranges': 'REDACTED'
    'ETag': '"0x8DEF4D059101874"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '869d61ac-f01e-0067-47b9-26c1ff000000'
    'x-ms-client-request-id': '74a1010a-92ac-11f1-be22-c8d9d2d2804a'
    'x-ms-version': 'REDACTED'


File size on Azure: 49961641 bytes


In [26]:
df_clean = df.copy()

df_clean["passenger_count"] = df_clean["passenger_count"].fillna(1)
df_clean["RatecodeID"] = df_clean["RatecodeID"].fillna(99)
df_clean["store_and_fwd_flag"] = df_clean["store_and_fwd_flag"].fillna("N")
df_clean["congestion_surcharge"] = df_clean["congestion_surcharge"].fillna(0)
df_clean["Airport_fee"] = df_clean["Airport_fee"].fillna(0)

df_clean = df_clean[df_clean["fare_amount"] > 0]
df_clean = df_clean[df_clean["trip_distance"] > 0]

print("Before cleaning:", len(df))
print("After cleaning:", len(df_clean))

Before cleaning: 2964624
After cleaning: 2869714


In [27]:
processed_file_system_client = service_client.get_file_system_client(file_system="processed")

processed_directory_client = processed_file_system_client.get_directory_client("nyc_taxi/2024/01")

processed_directory_client.create_directory()

print("Processed directory ready!")

2026-08-08 03:08:07,552 | INFO | Request URL: 'https://opspulseadl.dfs.core.windows.net/processed/nyc_taxi/2024/01?resource=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/json'
    'User-Agent': 'azsdk-python-storage-dfs/12.25.0 Python/3.13.4 (Windows-10-10.0.19045-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '784cb8a4-92ac-11f1-a8f4-c8d9d2d2804a'
    'Authorization': 'REDACTED'
No body was attached to the request
2026-08-08 03:08:08,194 | INFO | Response status: 201
Response headers:
    'Last-Modified': 'Fri, 07 Aug 2026 22:08:08 GMT'
    'ETag': '"0x8DEF4D05CD7244C"'
    'Server': 'Windows-Azure-HDFS/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-server-encrypted': 'REDACTED'
    'x-ms-request-id': '7db95ce0-901f-002c-7ab9-263dac000000'
    'x-ms-version': 'REDACTED'
    'x-ms-client-request-id': '784cb8a4-92ac-11f1-a8f4-c8d9d2d2804a'
    'Date': 'Fri, 07 Aug 2026 22:08:08 GMT'
    'Content-Length': '0'


Processed directory ready!


In [28]:
from io import BytesIO

processed_file_client = processed_directory_client.get_file_client("processed_nyc_taxi_2024_01.csv")

buffer = BytesIO()
df_clean.to_csv(buffer, index=False)
buffer.seek(0)

processed_file_client.upload_data(
    buffer,
    overwrite=True,
    connection_timeout=600,
    max_concurrency=1
)

print("Processed file uploaded!")

2026-08-08 03:09:15,990 | INFO | Request URL: 'https://opspulseadl.dfs.core.windows.net/processed/nyc_taxi/2024/01/processed_nyc_taxi_2024_01.csv?resource=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/json'
    'User-Agent': 'azsdk-python-storage-dfs/12.25.0 Python/3.13.4 (Windows-10-10.0.19045-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'a11777bf-92ac-11f1-a9ce-c8d9d2d2804a'
    'Authorization': 'REDACTED'
No body was attached to the request
2026-08-08 03:09:16,122 | INFO | Response status: 201
Response headers:
    'Last-Modified': 'Fri, 07 Aug 2026 22:09:16 GMT'
    'ETag': '"0x8DEF4D085A12C08"'
    'Server': 'Windows-Azure-HDFS/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-server-encrypted': 'REDACTED'
    'x-ms-request-id': '7db9633e-901f-002c-23b9-263dac000000'
    'x-ms-version': 'REDACTED'
    'x-ms-client-request-id': 'a11777bf-92ac-11f1-a9ce-c8d9d2d2804a'
    'Date': 'Fri, 07 Aug 2026 22:09:16 GMT

Processed file uploaded!


In [29]:
processed_properties = processed_file_client.get_file_properties()
print("Processed file size:", processed_properties.size, "bytes")

2026-08-08 03:15:10,594 | INFO | Request URL: 'https://opspulseadl.blob.core.windows.net/processed/nyc_taxi/2024/01/processed_nyc_taxi_2024_01.csv'
Request method: 'HEAD'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-dfs/12.25.0 Python/3.13.4 (Windows-10-10.0.19045-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '7473d12f-92ad-11f1-8a09-c8d9d2d2804a'
    'Authorization': 'REDACTED'
No body was attached to the request
2026-08-08 03:15:12,404 | INFO | Response status: 200
Response headers:
    'Content-Length': '305318302'
    'Content-Type': 'application/octet-stream'
    'Last-Modified': 'Fri, 07 Aug 2026 22:15:10 GMT'
    'Accept-Ranges': 'REDACTED'
    'ETag': '"0x8DEF4D158E22614"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '7d6ca1d5-601e-005a-5bba-26b7e4000000'
    'x-ms-client-request-id': '7473d12f-92ad-11f1-8a09-c8d9d2d2804a'
    'x-ms-version': 'REDA

Processed file size: 305318302 bytes


In [30]:
import json
import os

watermark_file = "watermark.json"

if os.path.exists(watermark_file):
    with open(watermark_file, "r") as f:
        last_watermark = json.load(f)["last_watermark"]
else:
    last_watermark = "1900-01-01 00:00:00"

print("Last watermark:", last_watermark)

Last watermark: NaT


In [31]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
last_watermark_dt = pd.to_datetime(last_watermark)

df_new = df[df["tpep_pickup_datetime"] > last_watermark_dt].copy()

print("Total rows in source:", len(df))
print("New rows since last watermark:", len(df_new))


Total rows in source: 2964624
New rows since last watermark: 0


In [32]:
if len(df_new) > 0:
    new_watermark = str(df_new["tpep_pickup_datetime"].max())
    with open("watermark.json", "w") as f:
        json.dump({"last_watermark": new_watermark}, f)
    print("Watermark saved:", new_watermark)
else:
    print("No new data, watermark unchanged.")

No new data, watermark unchanged.


In [33]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("etl_log.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("opspulse")

logger.info("Logging started!")

2026-08-08 03:15:12,924 | INFO | Logging started!


In [34]:
try:
    result = 10 / 0
    print(result)
except Exception as e:
    logger.error(f"Something went wrong: {e}")

2026-08-08 04:41:25,596 | ERROR | Something went wrong: division by zero


In [ ]:
import os
import json
import logging
import pandas as pd
from io import BytesIO
from azure.storage.filedatalake import DataLakeServiceClient

# ---------------- LOGGING ----------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler("etl_log.log"), logging.StreamHandler()]
)
logger = logging.getLogger("opspulse")

# ---------------- CONFIG ----------------
connection_string = "connection_string = "PASTE_YOUR_CONNECTION_STRING_HERE"
local_file_path = "C:/Users/TLS/Downloads/yellow_tripdata_2024-01.parquet"
watermark_file = "watermark.json"

try:
    # 1. Connect
    logger.info("Connecting to Azure...")
    service_client = DataLakeServiceClient.from_connection_string(connection_string)

    # 2. Read data
    logger.info("Reading source file...")
    df = pd.read_parquet(local_file_path)
    logger.info(f"Total rows: {len(df)}")

    # 3. Read watermark
    if os.path.exists(watermark_file):
        with open(watermark_file, "r") as f:
            last_watermark = json.load(f)["last_watermark"]
    else:
        last_watermark = "1900-01-01 00:00:00"
    logger.info(f"Last watermark: {last_watermark}")

    # 4. Filter new data only (incremental)
    df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
    df_new = df[df["tpep_pickup_datetime"] > pd.to_datetime(last_watermark)].copy()
    logger.info(f"New rows to process: {len(df_new)}")

    if len(df_new) == 0:
        logger.info("No new data. Stopping here.")
    else:
        # 5. Upload RAW
        raw_fs = service_client.get_file_system_client(file_system="raw")
        raw_dir = raw_fs.get_directory_client("nyc_taxi/2024/01")
        raw_dir.create_directory()
        raw_file = raw_dir.get_file_client("yellow_tripdata_2024-01.parquet")
        with open(local_file_path, "rb") as data:
            raw_file.upload_data(data, overwrite=True, connection_timeout=600, max_concurrency=1)
        logger.info("Raw file uploaded.")

        # 6. Transform
        df_clean = df_new.copy()
        df_clean["passenger_count"] = df_clean["passenger_count"].fillna(1)
        df_clean["RatecodeID"] = df_clean["RatecodeID"].fillna(99)
        df_clean["store_and_fwd_flag"] = df_clean["store_and_fwd_flag"].fillna("N")
        df_clean["congestion_surcharge"] = df_clean["congestion_surcharge"].fillna(0)
        df_clean["Airport_fee"] = df_clean["Airport_fee"].fillna(0)
        df_clean = df_clean[df_clean["fare_amount"] > 0]
        df_clean = df_clean[df_clean["trip_distance"] > 0]
        logger.info(f"Cleaned data: {len(df_clean)} rows")

        # 7. Upload PROCESSED
        processed_fs = service_client.get_file_system_client(file_system="processed")
        processed_dir = processed_fs.get_directory_client("nyc_taxi/2024/01")
        processed_dir.create_directory()
        processed_file = processed_dir.get_file_client("processed_nyc_taxi_2024_01.csv")
        buffer = BytesIO()
        df_clean.to_csv(buffer, index=False)
        buffer.seek(0)
        processed_file.upload_data(buffer, overwrite=True, connection_timeout=600, max_concurrency=1)
        logger.info("Processed file uploaded.")

        # 8. Update watermark (only if new data existed)
        new_watermark = str(df_new["tpep_pickup_datetime"].max())
        with open(watermark_file, "w") as f:
            json.dump({"last_watermark": new_watermark}, f)
        logger.info(f"Watermark updated: {new_watermark}")

    logger.info("Pipeline completed successfully.")

except Exception as e:
    logger.error(f"Pipeline failed: {e}")

2026-08-08 04:44:42,017 | INFO | Connecting to Azure...
2026-08-08 04:44:42,076 | INFO | Reading source file...
2026-08-08 04:44:50,469 | INFO | Total rows: 2964624
2026-08-08 04:44:50,552 | INFO | Last watermark: NaT
2026-08-08 04:44:50,886 | INFO | New rows to process: 0
2026-08-08 04:44:50,890 | INFO | No new data. Stopping here.
2026-08-08 04:44:50,894 | INFO | Pipeline completed successfully.
